# UniSRec Tutorial: Sequential Recommendation with Pretrained Text Embeddings

This tutorial demonstrates how to use **UniSRec** — a sequential recommender that leverages pretrained text embeddings (e.g. from sentence-transformers) instead of learned item ID embeddings.

## What is UniSRec?

UniSRec jointly trains a **PCA-based adaptor** and a **SASRec transformer encoder** on top of frozen pretrained text embeddings. This architecture:

- Enables **cold-start** recommendations for items with no interaction history (as long as text embeddings are available)
- Supports **cross-domain transfer** — a model trained on one domain can be applied to another if items share the same embedding space
- Achieves competitive quality with SASRec when high-quality text embeddings are used

## Architecture

```
frozen_text_emb  →  adaptor (PCA + MLP)  →  SASRec encoder  →  dot-product scoring
```

## What you will learn

1. How to prepare ML-20M data for UniSRec
2. How to generate (or load) pretrained item embeddings
3. How to train and evaluate UniSRec
4. How to use GPU-accelerated metrics
5. How to save/load checkpoints and export to ONNX
6. How to experiment with different losses and configurations

## Table of Contents

* [1. Setup](#1-setup)
* [2. Download and Prepare ML-20M](#2-download-and-prepare-ml-20m)
* [3. Pretrained Embeddings](#3-pretrained-embeddings)
* [4. Train UniSRec](#4-train-unisrec)
* [5. Evaluate](#5-evaluate)
* [6. Inference: Top-K Predictions](#6-inference-top-k-predictions)
* [7. Save / Load Checkpoints](#7-save--load-checkpoints)
* [8. ONNX Export](#8-onnx-export)
* [9. Experimenting with Configurations](#9-experimenting-with-configurations)
* [10. Key Parameters Reference](#10-key-parameters-reference)
* [Links](#links)

## 1. Setup

In [1]:
import io
import time
import warnings
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd
import requests
import torch
from tqdm import tqdm

from rectools.fast_transformers import UniSRecModel, compute_metrics
from rectools.fast_transformers.preprocessing import build_sequences

warnings.simplefilter("ignore")

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")
if DEVICE == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    torch.set_float32_matmul_precision("high")

Device: cuda
GPU: NVIDIA GeForce RTX 3090


## 2. Download and Prepare ML-20M

We use the [MovieLens 20M](https://grouplens.org/datasets/movielens/20m/) dataset — 20 million ratings from 138K users on 27K movies.

### Data preprocessing
- Filter items with fewer than 5 interactions
- Filter users with fewer than 2 interactions
- Leave-last-out split: last interaction per user → test, second-to-last → validation, rest → train

In [2]:
DATA_DIR = Path("data_ml20m")
RATINGS_PATH = DATA_DIR / "ratings.csv"
ML20M_URL = "https://files.grouplens.org/datasets/movielens/ml-20m.zip"

MIN_ITEM_INTERACTIONS = 5
MIN_USER_INTERACTIONS = 2


def download_ml20m():
    """Download and extract ML-20M if not present."""
    if RATINGS_PATH.exists():
        print(f"ML-20M already downloaded at {DATA_DIR}")
        return
    print(f"Downloading ML-20M from {ML20M_URL} ...")
    DATA_DIR.mkdir(parents=True, exist_ok=True)
    resp = requests.get(ML20M_URL, stream=True, timeout=600)
    resp.raise_for_status()
    buf = io.BytesIO()
    total = int(resp.headers.get("content-length", 0))
    with tqdm(total=total, unit="B", unit_scale=True, desc="Download") as pbar:
        for chunk in resp.iter_content(chunk_size=1 << 20):
            buf.write(chunk)
            pbar.update(len(chunk))
    print("Extracting...")
    with zipfile.ZipFile(buf) as zf:
        for member in zf.namelist():
            basename = Path(member).name
            if not basename:
                continue
            target = DATA_DIR / basename
            with zf.open(member) as src, open(target, "wb") as dst:
                dst.write(src.read())
    print(f"Extracted to {DATA_DIR}")


download_ml20m()

ML-20M already downloaded at data_ml20m


In [3]:
# Load and preprocess
ratings = pd.read_csv(RATINGS_PATH)
ratings.columns = ["user_id", "item_id", "rating", "timestamp"]

# Filter rare items
item_counts = ratings.groupby("item_id").size()
popular_items = item_counts[item_counts >= MIN_ITEM_INTERACTIONS].index
ratings = ratings[ratings["item_id"].isin(popular_items)]

# Filter rare users
user_counts = ratings.groupby("user_id").size()
active_users = user_counts[user_counts >= MIN_USER_INTERACTIONS].index
ratings = ratings[ratings["user_id"].isin(active_users)]

print(f"Interactions: {len(ratings):,}")
print(f"Users:        {ratings['user_id'].nunique():,}")
print(f"Items:        {ratings['item_id'].nunique():,}")

Interactions: 19,984,024
Users:        138,493
Items:        18,345


In [4]:
# Leave-last-out split
ratings = ratings.sort_values(["user_id", "timestamp"])

test = ratings.groupby("user_id").tail(1)
remaining = ratings.drop(test.index)
val = remaining.groupby("user_id").tail(1)
train = remaining.drop(val.index)

# For training we use train + val
train_with_val = pd.concat([train, val])

print(f"Train:      {len(train):,}")
print(f"Validation: {len(val):,}")
print(f"Test:       {len(test):,}")
print(f"Train+Val:  {len(train_with_val):,}")

Train:      19,707,038
Validation: 138,493
Test:       138,493
Train+Val:  19,845,531


In [5]:
# Convert to tensors — UniSRec works with raw PyTorch tensors
user_ids_t = torch.tensor(train_with_val["user_id"].values, dtype=torch.long)
item_ids_t = torch.tensor(train_with_val["item_id"].values, dtype=torch.long)
timestamps_t = torch.tensor(train_with_val["timestamp"].values, dtype=torch.long)

print(f"Tensor shapes: users={user_ids_t.shape}, items={item_ids_t.shape}, timestamps={timestamps_t.shape}")

Tensor shapes: users=torch.Size([19845531]), items=torch.Size([19845531]), timestamps=torch.Size([19845531])


## 3. Pretrained Embeddings

UniSRec requires a pretrained embedding matrix of shape `(max_external_item_id + 1, D_text)` where index `i` holds the text embedding for item with external ID `i`. Index 0 is padding (zeros).

**In production**, you would encode item descriptions with a sentence-transformer model (e.g., `all-MiniLM-L6-v2` or `Qwen3-Embedding-0.6B`). For this tutorial, we use **random embeddings** to demonstrate the API without requiring a GPU-heavy embedding step.

> **Tip:** With real text embeddings (e.g. from movie descriptions, genres, cast), UniSRec achieves much better quality. See the `demo_kion.md` guide for an example with Qwen3-Embedding.

In [6]:
EMB_DIM = 256  # pretrained embedding dimension

max_item_id = int(ratings["item_id"].max())
torch.manual_seed(42)
pretrained_embeddings = torch.randn(max_item_id + 1, EMB_DIM)
pretrained_embeddings[0] = 0.0  # padding row must be zeros

print(f"Pretrained embeddings shape: {pretrained_embeddings.shape}")
print(f"  - max_item_id: {max_item_id}")
print(f"  - embedding dim: {EMB_DIM}")

Pretrained embeddings shape: torch.Size([131014, 256])
  - max_item_id: 131013
  - embedding dim: 256


### How to use real text embeddings

If you have movie descriptions, here's how to generate real embeddings:

```python
from sentence_transformers import SentenceTransformer

encoder = SentenceTransformer("all-MiniLM-L6-v2")
movies = pd.read_csv(DATA_DIR / "movies.csv")  # from ML-20M

# Build text for each movie
texts = {}
for _, row in movies.iterrows():
    texts[row["movieId"]] = f"{row['title']}. Genres: {row['genres']}"

# Encode
max_id = movies["movieId"].max()
embeddings = torch.zeros(max_id + 1, 384)  # MiniLM outputs 384-dim
for item_id, text in texts.items():
    embeddings[item_id] = torch.tensor(encoder.encode(text))

torch.save(embeddings, "ml20m_embeddings.pt")
```

## 4. Train UniSRec

UniSRec accepts raw tensors `(user_ids, item_ids, timestamps)` — no need to construct a RecTools `Dataset`. Internally it:

1. Calls `build_sequences()` to create left-padded `(x, y)` sequence pairs on GPU
2. Remaps item IDs to contiguous `1..N` and aligns pretrained embeddings accordingly
3. Trains the adaptor + transformer with PyTorch Lightning

### Key hyperparameters

| Parameter | Description |
|-----------|-------------|
| `n_factors` | Hidden dimension of the transformer |
| `adaptor_type` | `"pca"` (default) or `"bn"` (batch-norm) |
| `session_max_len` | Maximum sequence length |
| `loss` | `"softmax"`, `"BCE"`, `"gBCE"`, `"sampled_softmax"` |
| `lr` | Base learning rate |
| `lr_transformer` | Multiplier for transformer layers (relative to `lr`) |
| `lr_head` | Multiplier for adaptor MLP head |

In [7]:
EPOCHS = 3  # use more epochs (10+) for better quality
N_FACTORS = 128
SESSION_MAX_LEN = 200
BATCH_SIZE = 128
LR = 1e-3

model = UniSRecModel(
    pretrained_item_embeddings=pretrained_embeddings,
    # Architecture
    n_factors=N_FACTORS,
    projection_hidden=N_FACTORS,
    n_blocks=2,
    n_heads=2,
    session_max_len=SESSION_MAX_LEN,
    dropout=0.1,
    adaptor_dropout=0.2,
    adaptor_type="pca",
    use_adaptor_ffn=True,
    ffn_type="conv1d",
    ffn_expansion=1,
    # Training
    epochs=EPOCHS,
    lr=LR,
    optimizer="adam",
    loss="softmax",
    grad_clip=1.0,
    weight_decay=0.0,
    batch_size=BATCH_SIZE,
    train_min_user_interactions=MIN_USER_INTERACTIONS,
    device=DEVICE,
    verbose=1,
)

print("Model created. Training...")

Model created. Training...


In [8]:
%%time
model.fit(user_ids_t, item_ids_t, timestamps_t)
print("Training complete!")

GPU available: True (cuda), used: True


TPU available: False, using: 0 TPU cores


HPU available: False, using: 0 HPUs


LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Training: |          | 0/? [00:00<?, ?it/s]

`Trainer.fit` stopped: `max_epochs=3` reached.


Training complete!
CPU times: user 2min 49s, sys: 1.01 s, total: 2min 50s
Wall time: 2min 48s


## 5. Evaluate

We evaluate using **leave-last-out** protocol: for each test user, predict the held-out item from the user's interaction history.

UniSRec provides GPU-accelerated metrics via `compute_metrics()` that stay entirely on device — no numpy/pandas roundtrips.

In [9]:
@torch.no_grad()
def evaluate_model(model, train_df, test_df, k=10, batch_size=256):
    """Evaluate UniSRec with leave-last-out protocol."""
    net = model.net
    net.eval().to(DEVICE)
    device = torch.device(DEVICE)
    maxlen = model.session_max_len

    # Get projected item embeddings (all items at once)
    item_embs = net.project_all()

    # Build external-to-internal ID mapping
    unique_items = model.item_id_mapping
    assert unique_items is not None
    ext_to_int = {int(unique_items[i].item()): i + 1 for i in range(len(unique_items))}

    # Group histories and targets
    train_grouped = train_df.sort_values("timestamp").groupby("user_id")["item_id"].agg(list).to_dict()
    test_grouped = test_df.groupby("user_id")["item_id"].first().to_dict()
    test_users = list(test_grouped.keys())

    all_topk, all_targets = [], []

    for start in tqdm(range(0, len(test_users), batch_size), desc="Evaluating"):
        batch_users = test_users[start:start + batch_size]
        seqs, targets = [], []

        for uid in batch_users:
            history = train_grouped.get(uid, [])
            mapped = [ext_to_int[iid] for iid in history if iid in ext_to_int]
            if not mapped:
                continue
            seq = mapped[-maxlen:]
            seqs.append([0] * (maxlen - len(seq)) + seq)
            target_int = ext_to_int.get(test_grouped[uid])
            targets.append(target_int if target_int is not None else 0)

        if not seqs:
            continue

        x = torch.tensor(seqs, dtype=torch.long, device=device)
        h = net.encode_last(x)
        scores = h @ item_embs.T
        scores[:, 0] = float("-inf")  # exclude padding token

        _, topk_ids = scores.topk(k, dim=1)
        all_topk.append(topk_ids)
        all_targets.append(torch.tensor(targets, dtype=torch.long, device=device))

    # Concatenate all batches and compute metrics
    topk_ids = torch.cat(all_topk, dim=0)
    targets = torch.cat(all_targets, dim=0)

    return compute_metrics(topk_ids, targets, ks=[5, 10])

In [10]:
%%time
metrics = evaluate_model(model, train_with_val, test)

print("\nEvaluation Results:")
print("-" * 35)
for name, value in metrics.items():
    print(f"  {name:10s} = {value:.4f}")


Evaluating:   0%|          | 0/541 [00:00<?, ?it/s]


Evaluating:   1%|          | 6/541 [00:00<00:09, 56.89it/s]


Evaluating:   3%|▎         | 14/541 [00:00<00:07, 67.42it/s]


Evaluating:   4%|▍         | 22/541 [00:00<00:07, 70.79it/s]


Evaluating:   6%|▌         | 30/541 [00:00<00:06, 73.23it/s]

...


Evaluating:  96%|█████████▌| 518/541 [00:06<00:00, 76.07it/s]


Evaluating:  97%|█████████▋| 526/541 [00:06<00:00, 76.34it/s]


Evaluating:  99%|█████████▊| 534/541 [00:07<00:00, 76.38it/s]


Evaluating: 100%|██████████| 541/541 [00:07<00:00, 75.75it/s]


Evaluation Results:
-----------------------------------
  HR@5       = 0.0635
  NDCG@5     = 0.0143
  MRR@5      = 0.0352
  HR@10      = 0.0972
  NDCG@10    = 0.0117
  MRR@10     = 0.0396
CPU times: user 12.7 s, sys: 512 ms, total: 13.2 s
Wall time: 13.1 s


## 6. Inference: Top-K Predictions

UniSRec provides `predict_topk()` for direct GPU inference. It fuses sequence encoding and dot-product ranking in a single call.

You can also use `map_item_ids()` to convert external IDs to internal ones.

In [11]:
# Pick a test user and build their sequence
test_user_id = test["user_id"].iloc[0]
user_history = (
    train_with_val[train_with_val["user_id"] == test_user_id]
    .sort_values("timestamp")["item_id"]
    .tolist()
)
print(f"User {test_user_id}: {len(user_history)} interactions in history")

# Map external item IDs to internal IDs
external_ids = torch.tensor(user_history, dtype=torch.long)
internal_ids = model.map_item_ids(external_ids)

# Build left-padded sequence
seq = internal_ids[-SESSION_MAX_LEN:]
padded = torch.zeros(SESSION_MAX_LEN, dtype=torch.long)
padded[-len(seq):] = seq
input_ids = padded.unsqueeze(0).to(DEVICE)  # (1, max_len)

# Get top-5 predictions
scores, top_ids = model.predict_topk(input_ids, k=5)

print(f"\nTop-5 predictions (internal IDs): {top_ids[0].cpu().tolist()}")
print(f"Scores: {scores[0].cpu().tolist()}")

# Map back to external IDs
unique_items = model.item_id_mapping
assert unique_items is not None
external_recs = [int(unique_items[iid - 1].item()) for iid in top_ids[0].cpu().tolist()]
print(f"Top-5 predictions (external IDs): {external_recs}")

# Check ground truth
gt_item = test[test["user_id"] == test_user_id]["item_id"].iloc[0]
print(f"\nGround truth item: {gt_item}")
print(f"Hit: {'Yes' if gt_item in external_recs else 'No'}")

User 1: 174 interactions in history

Top-5 predictions (internal IDs): [316, 294, 353, 4143, 9937]
Scores: [1.828413486480713, 1.4550416469573975, 1.4328746795654297, 1.3565888404846191, 1.281807541847229]
Top-5 predictions (external IDs): [318, 296, 356, 4246, 34162]

Ground truth item: 1750
Hit: No


## 7. Save / Load Checkpoints

Save the trained model for later use. On load, the pretrained embeddings are realigned automatically.

In [12]:
CKPT_PATH = "unisrec_ml20m.pt"

# Save
model.save_checkpoint(CKPT_PATH)
print(f"Checkpoint saved to {CKPT_PATH}")

# Load into a new model instance
model_loaded = UniSRecModel(
    pretrained_item_embeddings=pretrained_embeddings,
    n_factors=N_FACTORS,
    projection_hidden=N_FACTORS,
    n_blocks=2,
    n_heads=2,
    session_max_len=SESSION_MAX_LEN,
)
model_loaded.load_checkpoint(CKPT_PATH, device=DEVICE)
print(f"Checkpoint loaded. is_fitted={model_loaded.is_fitted}")

# Verify predictions match
scores_orig, ids_orig = model.predict_topk(input_ids, k=5)
scores_loaded, ids_loaded = model_loaded.predict_topk(input_ids, k=5)
print(f"Predictions match: {torch.equal(ids_orig, ids_loaded)}")

Checkpoint saved to unisrec_ml20m.pt


Checkpoint loaded. is_fitted=True
Predictions match: True


## 8. ONNX Export

Export the model to ONNX for deployment. Two graphs are exported:
- **Encoder**: `input_ids → hidden states`
- **Items**: `→ item embeddings` (all projected items)

In [13]:
try:
    ENCODER_ONNX = "unisrec_encoder.onnx"
    ITEMS_ONNX = "unisrec_items.onnx"

    model.export_to_onnx(
        encoder_path=ENCODER_ONNX,
        items_path=ITEMS_ONNX,
    )
    print(f"Exported encoder to {ENCODER_ONNX}")
    print(f"Exported items to {ITEMS_ONNX}")

    # Show file sizes
    for p in [ENCODER_ONNX, ITEMS_ONNX]:
        size_mb = Path(p).stat().st_size / (1024 * 1024)
        print(f"  {p}: {size_mb:.1f} MB")
except ImportError as e:
    print(f"ONNX export skipped — missing dependency: {e}")
    print("Install with: pip install onnx onnxscript onnxruntime")

ONNX export skipped — missing dependency: No module named 'onnxscript'
Install with: pip install onnx onnxscript onnxruntime


## 9. Experimenting with Configurations

UniSRec supports multiple loss functions, adaptor types, and FFN architectures. Let's compare a few configurations.

### 9.1 Loss functions

UniSRec supports 4 loss functions:

| Loss | Description | Requires `n_negatives` |
|------|-------------|----------------------|
| `softmax` | Cross-entropy over full item catalog (most precise, slower) | No |
| `sampled_softmax` | Cross-entropy over sampled negatives | Yes |
| `BCE` | Binary cross-entropy with negative sampling | Yes |
| `gBCE` | Calibrated BCE that reduces overconfidence ([gSASRec paper](https://arxiv.org/abs/2308.07192)) | Yes |

In [14]:
# Example: BCE loss with 50 negatives per positive
model_bce = UniSRecModel(
    pretrained_item_embeddings=pretrained_embeddings,
    n_factors=N_FACTORS,
    projection_hidden=N_FACTORS,
    n_blocks=2,
    n_heads=2,
    session_max_len=SESSION_MAX_LEN,
    loss="BCE",
    n_negatives=50,
    epochs=EPOCHS,
    lr=LR,
    optimizer="adam",
    batch_size=BATCH_SIZE,
    device=DEVICE,
    verbose=1,
)

print("Training with BCE loss...")
model_bce.fit(user_ids_t, item_ids_t, timestamps_t)

metrics_bce = evaluate_model(model_bce, train_with_val, test)
print("\nBCE Results:")
for name, value in metrics_bce.items():
    print(f"  {name:10s} = {value:.4f}")

Training with BCE loss...


GPU available: True (cuda), used: True


TPU available: False, using: 0 TPU cores


HPU available: False, using: 0 HPUs


LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


...


Evaluating:  96%|█████████▌| 520/541 [00:06<00:00, 76.17it/s]


Evaluating:  98%|█████████▊| 528/541 [00:06<00:00, 76.03it/s]


Evaluating:  99%|█████████▉| 536/541 [00:07<00:00, 76.25it/s]


Evaluating: 100%|██████████| 541/541 [00:07<00:00, 75.80it/s]


BCE Results:
  HR@5       = 0.0164
  NDCG@5     = 0.0032
  MRR@5      = 0.0072
  HR@10      = 0.0288
  NDCG@10    = 0.0030
  MRR@10     = 0.0088


### 9.2 Adaptor types

UniSRec supports two adaptor architectures for projecting pretrained embeddings:

| Type | Description |
|------|-------------|
| `"pca"` | PCA whitening + optional MLP head (default, recommended) |
| `"bn"` | Batch normalization + MLP head |

In [15]:
# Example: batch-norm adaptor
model_bn = UniSRecModel(
    pretrained_item_embeddings=pretrained_embeddings,
    n_factors=N_FACTORS,
    projection_hidden=N_FACTORS,
    n_blocks=2,
    n_heads=2,
    session_max_len=SESSION_MAX_LEN,
    adaptor_type="bn",  # batch-norm instead of PCA
    loss="softmax",
    epochs=EPOCHS,
    lr=LR,
    optimizer="adam",
    batch_size=BATCH_SIZE,
    device=DEVICE,
    verbose=1,
)

print("Training with BN adaptor...")
model_bn.fit(user_ids_t, item_ids_t, timestamps_t)

metrics_bn = evaluate_model(model_bn, train_with_val, test)
print("\nBN Adaptor Results:")
for name, value in metrics_bn.items():
    print(f"  {name:10s} = {value:.4f}")

Training with BN adaptor...


GPU available: True (cuda), used: True


TPU available: False, using: 0 TPU cores


HPU available: False, using: 0 HPUs


LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


...


Evaluating:  96%|█████████▌| 520/541 [00:06<00:00, 76.55it/s]


Evaluating:  98%|█████████▊| 528/541 [00:06<00:00, 76.42it/s]


Evaluating:  99%|█████████▉| 536/541 [00:07<00:00, 76.65it/s]


Evaluating: 100%|██████████| 541/541 [00:07<00:00, 76.47it/s]


BN Adaptor Results:
  HR@5       = 0.0309
  NDCG@5     = 0.0062
  MRR@5      = 0.0141
  HR@10      = 0.0592
  NDCG@10    = 0.0060
  MRR@10     = 0.0178


### 9.3 Advanced training: LR scheduler + early stopping

In [16]:
# Cosine warmup scheduler + early stopping
model_advanced = UniSRecModel(
    pretrained_item_embeddings=pretrained_embeddings,
    n_factors=N_FACTORS,
    projection_hidden=N_FACTORS,
    n_blocks=2,
    n_heads=2,
    session_max_len=SESSION_MAX_LEN,
    # Training
    loss="softmax",
    epochs=10,
    lr=1e-4,
    optimizer="adamw",
    weight_decay=0.01,
    # LR scheduler
    scheduler="cosine_warmup",
    warmup_ratio=0.05,
    min_lr_ratio=0.1,
    # Early stopping on validation loss
    patience=3,
    batch_size=BATCH_SIZE,
    device=DEVICE,
    verbose=1,
)

print("Training with cosine warmup + early stopping...")
model_advanced.fit(user_ids_t, item_ids_t, timestamps_t)

metrics_adv = evaluate_model(model_advanced, train_with_val, test)
print("\nAdvanced Training Results:")
for name, value in metrics_adv.items():
    print(f"  {name:10s} = {value:.4f}")

Training with cosine warmup + early stopping...


GPU available: True (cuda), used: True


TPU available: False, using: 0 TPU cores


HPU available: False, using: 0 HPUs


LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


...


Evaluating:  96%|█████████▌| 520/541 [00:06<00:00, 75.62it/s]


Evaluating:  98%|█████████▊| 528/541 [00:06<00:00, 75.63it/s]


Evaluating:  99%|█████████▉| 536/541 [00:07<00:00, 75.91it/s]


Evaluating: 100%|██████████| 541/541 [00:07<00:00, 75.55it/s]


Advanced Training Results:
  HR@5       = 0.0624
  NDCG@5     = 0.0144
  MRR@5      = 0.0358
  HR@10      = 0.0924
  NDCG@10    = 0.0114
  MRR@10     = 0.0398


### 9.4 Results comparison

In [17]:
results = {
    "Softmax (PCA)": metrics,
    "BCE (PCA)": metrics_bce,
    "Softmax (BN)": metrics_bn,
    "AdamW + Cosine": metrics_adv,
}

comparison = pd.DataFrame(results).T
comparison.index.name = "Configuration"
print(comparison.to_string(float_format="%.4f"))

                 HR@5  NDCG@5  MRR@5  HR@10  NDCG@10  MRR@10
Configuration                                               
Softmax (PCA)  0.0635  0.0143 0.0352 0.0972   0.0117  0.0396
BCE (PCA)      0.0164  0.0032 0.0072 0.0288   0.0030  0.0088
Softmax (BN)   0.0309  0.0062 0.0141 0.0592   0.0060  0.0178
AdamW + Cosine 0.0624  0.0144 0.0358 0.0924   0.0114  0.0398


## 10. Key Parameters Reference

| Parameter | Description | Default |
|-----------|-------------|---------|
| **Architecture** | | |
| `n_factors` | Hidden dimension of the transformer | 256 |
| `projection_hidden` | Intermediate dim for PCA adaptor head | 512 |
| `n_blocks` | Number of transformer blocks | 2 |
| `n_heads` | Number of attention heads | 1 |
| `session_max_len` | Maximum sequence length | 200 |
| `dropout` | Dropout in transformer blocks | 0.1 |
| `adaptor_dropout` | Dropout in adaptor MLP | 0.2 |
| `adaptor_type` | `"pca"` or `"bn"` | `"pca"` |
| `use_adaptor_ffn` | Use MLP head after projection | `True` |
| `ffn_type` | `"conv1d"`, `"linear_gelu"`, `"linear_relu"` | `"conv1d"` |
| `ffn_expansion` | FFN hidden dim multiplier | 1 |
| **Training** | | |
| `epochs` | Number of training epochs | 10 |
| `lr` | Base learning rate | 1e-4 |
| `lr_head` | LR multiplier for adaptor head | 0.3 |
| `lr_wp` | LR multiplier for whitening projection | 0.1 |
| `lr_transformer` | LR multiplier for transformer layers | 3.0 |
| `optimizer` | `"adam"` or `"adamw"` | `"adamw"` |
| `scheduler` | `None` or `"cosine_warmup"` | `None` |
| `grad_clip` | Gradient clipping value | 1.0 |
| `weight_decay` | Weight decay | 0.01 |
| **Loss** | | |
| `loss` | `"softmax"`, `"BCE"`, `"gBCE"`, `"sampled_softmax"` | `"softmax"` |
| `n_negatives` | Number of negatives (for BCE/gBCE/sampled_softmax) | `None` |
| `gbce_t` | gBCE calibration temperature | 0.2 |
| **Data** | | |
| `batch_size` | Training batch size | 128 |
| `train_min_user_interactions` | Min interactions to include a user | 2 |
| `patience` | Early stopping patience (`None` = disabled) | `None` |

## Links

1. **UniSRec original paper:** [Towards Universal Sequence Representation Learning for Recommender Systems](https://arxiv.org/abs/2206.05941)
2. **SASRec:** [Self-Attentive Sequential Recommendation](https://arxiv.org/abs/1808.09781)
3. **gBCE loss:** [gSASRec: Reducing Overconfidence in Sequential Recommendation Trained with Negative Sampling](https://arxiv.org/abs/2308.07192)
4. **RecTools SASRec tutorial:** [transformers_tutorial.ipynb](./transformers_tutorial.ipynb)
5. **RecTools advanced training:** [transformers_advanced_training_guide.ipynb](./transformers_advanced_training_guide.ipynb)

In [18]:
# Cleanup temporary files
import os
for f in ["unisrec_ml20m.pt", "unisrec_encoder.onnx", "unisrec_items.onnx"]:
    if os.path.exists(f):
        os.remove(f)
print("Cleanup done.")

Cleanup done.
